In [73]:
import pandas as pd
import os
import requests

from rdkit import Chem
from pubchempy import get_compounds
from Bio.PDB.MMCIF2Dict import MMCIF2Dict
from Bio.PDB.PDBList import PDBList

In [ ]:
df = pd.read_excel('../data/row/pdbbind_NL_cleaned.xlsx')

In [ ]:
def extract_title(data):
    """Return the PDB structure title.""" 
    return data['_struct.title']


def extract_and_filter_sequences(pdb_dict):
    """
    Extracts nucleotide sequences, filters for valid characters (A, G, C, T, U),
    and returns a single sequence if all are identical, or a comma-separated
    string of unique sequences.

    Args:
        pdb_dict (dict): A dictionary containing PDB structure data.

    Returns:
        str: Single sequence if all valid sequences are identical, comma-separated
             string of unique sequences if different, or None if no valid data found.
    """

    nucleotide_sequences = {}

    if '_pdbx_poly_seq_scheme.asym_id' in pdb_dict and '_pdbx_poly_seq_scheme.mon_id' in pdb_dict and '_pdbx_poly_seq_scheme.pdb_strand_id' in pdb_dict:
        asym_ids = pdb_dict['_pdbx_poly_seq_scheme.asym_id']
        mon_ids = pdb_dict['_pdbx_poly_seq_scheme.mon_id']
        strand_ids = pdb_dict['_pdbx_poly_seq_scheme.pdb_strand_id']

        # Group monomers by strand ID
        strand_sequences = {}
        for i in range(len(asym_ids)):
            strand_id = strand_ids[i]
            monomer_id = mon_ids[i]
            if strand_id not in strand_sequences:
                strand_sequences[strand_id] = []
            strand_sequences[strand_id].append(monomer_id)

        # Concatenate monomers to form the nucleotide sequence for each strand
        for strand_id, sequence_list in strand_sequences.items():
            processed_sequence = ''.join([monomer[1:] if monomer.startswith('D') else monomer for monomer in sequence_list])
            processed_sequence = processed_sequence.replace('5IU', 'U')
            nucleotide_sequences[strand_id] = processed_sequence

        # Filter for valid characters and store
        valid_sequences = {}
        for strand, sequence in nucleotide_sequences.items():
            filtered_sequence = ''.join(c for c in sequence if c in 'AGCTU')
            if filtered_sequence:  # Only store if the sequence isn't empty after filtering
                valid_sequences[strand] = filtered_sequence

        if not valid_sequences:
            return None

        # Check if all valid sequences are the same
        first_sequence = next(iter(valid_sequences.values()), None)
        if first_sequence is not None and all(sequence == first_sequence for sequence in valid_sequences.values()):
            return first_sequence  # Return the single sequence if all are the same
        else:
            # Otherwise, return a comma-separated string of unique sequences
            unique_sequences = sorted(list(set(valid_sequences.values()))) # Find unique sequences and sort for consistent output
            return ','.join(unique_sequences)
    else:
        return None
    

def converter(lig_name, download_path="."):
    """
    Download an SDF file for a given ligand from the RCSB database,
    convert it to a SMILES string, and return the result.

    Parameters:
        lig_name (str): Ligand name/code (e.g., 'ATP').
        download_path (str): Directory where the SDF file will be saved.
                             Defaults to current directory.

    Returns:
        str or None: SMILES string if conversion is successful,
                     otherwise None.
    """
    
    os.makedirs(download_path, exist_ok=True)
    url = f"https://files.rcsb.org/ligands/download/{lig_name}_ideal.sdf"
    
    response = requests.get(url)
    file_name = os.path.join(download_path, f"{lig_name}_ideal.sdf")
    
    with open(file_name, 'wb') as file:
        file.write(response.content)

    sdf_file = Chem.SDMolSupplier(file_name)
    
    for mol in sdf_file:
        if mol is not None:
            return Chem.MolToSmiles(mol)

    return None
  

In [ ]:
#Download structures

pdbl = PDBList()
pdb_id_list = df.loc[:, 'pdb'].str.strip().to_list()
directory = '../data/row/pdb/'
for n in range(len(pdb_id_list)):
        try:
            pdb = pdb_id_list[n].upper()
            pdb_file = pdbl.download_pdb_files(pdb_id_list, pdir=directory)
        except Exception as e:
              continue

In [ ]:
for i in df.index:

    pdb_name = df.loc[i, 'pdb']
    lig_name = df.loc[i, 'ligand']
    pdb_dict = MMCIF2Dict(f"{directory}/{pdb_name}.cif")
    df.loc[i, 'content'] = extract_and_filter_sequences(pdb_dict)  
    smile = converter(lig_name, '../data/row/sdf_files')
    df.loc[i, 'smiles'] = smile
    df.loc[i, 'annotation'] = extract_title(pdb_dict)
    # get ligand name
    compounds = get_compounds(smile, namespace='smiles')
    for compound in compounds:
        iupac_name = compound.iupac_name
        df.loc[i, 'ligand_name'] = iupac_name
    

In [103]:
update_idx = [0, 1, 3]
df.loc[update_idx, 'content'] = 'GAAGCTTC'
drop_idx = [
    12,   # RNA pseudoknot
    18,   # Two-Base Bulge Site in DNA
    21,   # Bulged DNA
    105,  # DNA Bis-intercalating Anticancer Drug
    6,    # DNA Duplex
    136,  # not an aptamer
    28,   # ribozyme
    52,   # ribosomal RNA
    53, 54, 55,  # riboswitch
    95    # riboswitch
]

df.drop(index=drop_idx, inplace=True)

In [ ]:
df['content'] = df['content'].apply(lambda x: x.split(','))
df = df.explode('content')
df.drop(0, inplace=True)

In [ ]:
df.to_csv('../data/row/df_PDBBind_processed.csv')